# 03. 이상 탐지 모델 비교 (F1)

| 항목 | 내용 |
| --- | --- |
| 바꿀 곳 | 1번 셀의 `DATA_FILE`, `ZONE` 두 줄 (입력 컬럼 묶음은 `USE_`로 시작하는 아홉 줄) |
| v6 입력 | 기본 + 공통 움직임(구역 그룹별) + 10분 이동통계 + 지난 값 Z + OP 공통 움직임 + 로 전체 동시 이상 + 제어 어긋남. 포화비율, 경과시간, 변화율만 꺼짐. v1 ~ v5 결과를 다시 만들 때는 `USE_`를 모두 False로 |
| 입력 | `data/processed/cal01_base_v1.csv` ~ `cal01_base_v6.csv` 중 하나 |
| 출력 | `data/processed/model_results/` 안의 CSV 3개 (실행할 때마다 덮어씀) |

**LABEL 만드는 규칙** (5번 셀 한 줄)

| DOMAIN_LABEL | STAT_LABEL | LABEL |
| --- | --- | --- |
| 1 | 상관없음 | 1 (팀이 이상으로 확정) |
| 0 | 1 | 0 (팀이 검토해서 해제) |
| 빈칸 | 1 | 1 (통계 후보) |
| 빈칸 | 0 | 0 |
| 빈칸 | 빈칸 | 평가에서 뺌 |

**비교하는 4가지**

| 이름 | 하는 일 | 왜 보나 |
| --- | --- | --- |
| Dummy | 전부 0으로 예측 | 바닥 점수. F1은 항상 0 |
| Z-score | 같은 LOT에서 **지금까지 본 값만으로** \|Z\| > 3이면 1 | 가장 단순한 규칙 |
| IsolationForest | 라벨 없이 평소와 다른 점을 찾음 | 정답 없이 어디까지 찾나 |
| RandomForest | 라벨을 배워서 예측 | 팀 판단을 따라 하나 |

- STAT_LABEL은 LOT 전체(미래 값 포함)의 평균으로 만든 것임. 같은 방법으로 맞히면 F1이 1이 되므로 Z-score는 지난 값만 씀

**평가 방법** (시간 순서 유지)

- LOT 번호 순서가 시간 순서임
- 앞의 LOT들로 학습하고, 바로 다음 LOT 하나로 시험함
- 첫 4개 LOT은 학습에만 쓰고, 5번째부터 14번째 LOT까지 10번 시험함
- 10번 시험한 결과를 모두 모아 F1을 계산함

## 1. 공통 경로와 설정 (`DATA_FILE`, `ZONE` 두 줄만 바꿈)

In [ ]:
import os
import pandas as pd

# 저장소 루트 찾기: notebooks 폴더가 안 보이면 한 칸 위로 (최대 두 번)
ROOT = os.getcwd()
for step in range(2):
    if not os.path.isdir(os.path.join(ROOT, "notebooks")):
        ROOT = os.path.dirname(ROOT)

# 폴더
DATA_DIR = os.path.join(ROOT, "data")                    # 원본 CSV 8개
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")             # data/raw에 두었으면 그 폴더를 씀
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")  # 만든 CSV (cal01_base_v1 ~ v6)
LABEL_DIR = os.path.join(ROOT, "data", "labels")         # 팀 검토 표 (domain_decisions.csv)
FIG_DIR = os.path.join(ROOT, "figures")                  # 그림
ENC = "utf-8-sig"

# 원본 CSV
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")

# 읽을 가공 CSV (이름만 바꾸면 됨)
DATA_FILE = "cal01_base_v6.csv"  # cal01_base_v1.csv ~ cal01_base_v6.csv
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)

# 비교할 구역
ZONE = "Z4"  # CLN, Z1 ~ Z9, COOLING

# 입력 컬럼 묶음 (v6 입력은 포화비율, 경과시간, 변화율만 꺼짐)
# 공통 움직임 컬럼 (Z3 ~ Z6 온도가 최근 5분 사이에 같이 움직였는지)을 넣을지
USE_COMMON = True

# 수치 컬럼 묶음 (모두 지난 값만 사용. 포화비율, 경과시간, 변화율은 v6 실험에서 효과가 없어 꺼 둠)
USE_SAT = False      # OP가 끝(99% 이상, 5% 이하)에 닿은 비율, 최근 10분
USE_ELAPSED = False  # LOT 첫 기록부터 지난 분
USE_ROLL10 = True   # 직전 10분 평균과의 차이, 직전 10분 표준편차
USE_RATE = False     # 최근 5분 동안 1분에 평균 얼마나 변했는지
USE_ZPAST = True    # 지난 값만으로 계산한 |Z| (Z-score 기준선과 같은 계산)
USE_OPCOMMON = True # Z3 ~ Z6 출력(OP)이 최근 5분 사이에 같이 움직였는지
USE_FURNACE = True  # Z1 ~ Z8 중 최근 5분 안에 같이 튄 구역 수
USE_CONTROL = True  # 최근 5분 동안 내 구역 TC와 OP가 같은 방향으로 움직인 비율

## 2. 라이브러리와 저장 폴더

In [ ]:
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import IsolationForest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RESULT_DIR = os.path.join(PROCESSED_DIR, "model_results")
os.makedirs(RESULT_DIR, exist_ok=True)

# v4 ~ v6은 1분 묶음 데이터
MINUTE_DATA = DATA_FILE in ["cal01_base_v4.csv", "cal01_base_v5.csv", "cal01_base_v6.csv"]
print(DATA_FILE, "/ 1분 데이터:", MINUTE_DATA)

## 3. 구역과 센서

In [ ]:
# 구역별 센서 (CP-Z4M은 값이 0.0000000001 수준이라 라벨 계산에 쓰지 않음)
GROUPS = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
    "Z9": ["TC-Z9"],
    "COOLING": ["TC-OUT1", "TC-OUT2"],
}

# 라벨 계산에 쓰는 센서 21개
SENSORS = []
for zone in GROUPS:
    SENSORS = SENSORS + GROUPS[zone]

# CSV에 저장하는 원본 센서 22개 (CP-Z4M은 저장만 하고 계산에는 안 씀)
SAVE_SENSORS = SENSORS + ["CP-Z4M"]

# 라벨 컬럼 22개 (구역마다 STAT, DOMAIN)
LABEL_COLUMNS = []
for zone in GROUPS:
    LABEL_COLUMNS.append(zone + "__STAT_LABEL")
    LABEL_COLUMNS.append(zone + "__DOMAIN_LABEL")

sensors = GROUPS[ZONE]
print(ZONE, "센서:", sensors)

# 공통 움직임 그룹: 같이 움직이는 구역끼리 묶음 (5분 변화 상관: Z1 → Z2 0.88 (2분 뒤), Z3 → Z4 0.51 (4분 뒤), Z4 ↔ Z5 0.62, Z7 ↔ Z8 0.19)
# CLN, Z9, COOLING은 다른 구역과 상관이 0.3 미만이라 그룹이 없음
COMMON_GROUPS = {
    "Z1": ["Z1", "Z2"], "Z2": ["Z1", "Z2"],                                   # 건조, 예열
    "Z3": ["Z3", "Z4", "Z5", "Z6"], "Z4": ["Z3", "Z4", "Z5", "Z6"],           # 가열, 균열
    "Z5": ["Z3", "Z4", "Z5", "Z6"], "Z6": ["Z3", "Z4", "Z5", "Z6"],
    "Z7": ["Z7", "Z8"], "Z8": ["Z7", "Z8"],                                   # 과시효
}

# 내 구역이 속한 그룹의 온도와 출력 (그룹이 없으면 빈 목록이라 공통 움직임 컬럼이 안 생김)
COMMON_SENSORS = []
COMMON_OPS = []
for zone in COMMON_GROUPS.get(ZONE, []):
    COMMON_SENSORS.append("TC-" + zone)
    COMMON_OPS.append("OP-" + zone)
print("공통 움직임 그룹:", COMMON_SENSORS)

# 로 전체 동시 이상 수에 쓰는 구역 (건조대 Z1, Z2부터 과시효대 Z7, Z8까지)
FURNACE_ZONES = ["Z1", "Z2", "Z3", "Z4", "Z5", "Z6", "Z7", "Z8"]

## 4. 데이터 읽기 (필요한 컬럼만)

In [ ]:
columns = ["LOT_NO", "MEAS_DT", "EXCLUDED", ZONE + "__STAT_LABEL", ZONE + "__DOMAIN_LABEL"]

for sensor in sensors:
    columns.append(sensor)

    if MINUTE_DATA:
        columns = columns + [sensor + "__MIN", sensor + "__MAX", sensor + "__STD"]

# 공통 움직임 컬럼을 쓰면 Z3 ~ Z6 온도도 읽음
if USE_COMMON:
    for sensor in COMMON_SENSORS:
        if sensor not in columns:
            columns.append(sensor)

if USE_OPCOMMON:
    for sensor in COMMON_OPS:
        if sensor not in columns:
            columns.append(sensor)

if USE_FURNACE:
    for zone in FURNACE_ZONES:
        for sensor in GROUPS[zone]:
            if sensor not in columns:
                columns.append(sensor)

df = pd.read_csv(BASE_CSV, encoding=ENC, usecols=columns)
df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"], errors="raise")
df = df.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)

# LOT 첫 기록부터 지난 분 (USE_ELAPSED일 때 입력으로 씀)
lot_start = df.groupby("LOT_NO")["MEAS_DT"].transform("min")
df["LOT__ELAPSED"] = (df["MEAS_DT"] - lot_start).dt.total_seconds() / 60

print(df.shape)

## 5. LABEL 만들기

도메인 판단이 있으면 도메인 값, 없으면 통계 후보 값을 씀. 제외 행과 라벨이 빈 행은 뺌.

In [ ]:
df["LABEL"] = df[ZONE + "__DOMAIN_LABEL"].fillna(df[ZONE + "__STAT_LABEL"])

# 다른 규칙을 쓰려면 위 한 줄만 바꿈
# 예) 팀이 1로 확정한 것만 1로 볼 때
# df["LABEL"] = (df[ZONE + "__DOMAIN_LABEL"] == 1).astype(int).where(df[ZONE + "__STAT_LABEL"].notna())

df = df[(df["EXCLUDED"] == False) & df["LABEL"].notna()].reset_index(drop=True)
df["LABEL"] = df["LABEL"].astype(int)

print("평가할 행:", len(df))
print("LABEL 1 비율:", round(df["LABEL"].mean() * 100, 2), "%")

df.groupby("LOT_NO")["LABEL"].agg(["count", "sum"])

## 6. 입력 변수 만들기

| 변수 | 뜻 | 원기록 (v1 ~ v3) | 1분 (v4 ~ v6) |
| --- | --- | --- | --- |
| 센서 | 현재 값 | O | O (1분 평균) |
| `__DIFF` | 바로 앞 행과의 차이 | O | O |
| `__DEV60` | 직전 60행 평균과의 차이 | O | |
| `__STD60` | 직전 60행 표준편차 | O | |
| `__MIN`, `__MAX`, `__STD` | 1분 안의 최소, 최대, 표준편차 | | O |
| `온도__CHG5` | 내 구역 그룹(Z1·Z2, Z3~Z6, Z7·Z8) 온도의 5분 전 대비 변화 (`USE_COMMON`일 때) | O | O |
| `COMMON__OTHERS` | 내 구역 말고 같은 그룹 온도 중 최근 5분 안에 \|Z\| > 3이었던 수 (`USE_COMMON`일 때) | O | O |
| `OP__SAT10` | 최근 10분 중 OP가 99% 이상이나 5% 이하에 닿은 비율 (`USE_SAT`일 때) | O | O (1분 최대, 최소로 판단) |
| `LOT__ELAPSED` | LOT 첫 기록부터 지난 분 (`USE_ELAPSED`일 때) | O | O |
| `__DEV10`, `__STD10` | 직전 10분 평균과의 차이, 직전 10분 표준편차 (`USE_ROLL10`일 때) | O | O |
| `__RATE5` | 최근 5분 동안 1분에 평균 얼마나 변했는지 (`USE_RATE`일 때) | O | O |
| `__ZPAST` | 지난 값만으로 계산한 \|Z\| (`USE_ZPAST`일 때). 7번 Z-score 기준선과 같은 계산 | O | O |
| `__ZPAST_EDGE` | 1분 안의 최대, 최소 중 지난 평균에서 더 먼 쪽의 \|Z\| (`USE_ZPAST`일 때) | | O |
| `OP__CHG5`, `COMMON__OP_OTHERS` | 같은 그룹 출력의 5분 전 대비 변화, 내 구역 말고 최근 5분 안에 \|Z\| > 3이었던 출력 수 (`USE_OPCOMMON`일 때) | O | O |
| `FURNACE__OTHERS` | Z1 ~ Z8 중 내 구역 말고 최근 5분 안에 센서 하나라도 \|Z\| > 3이었던 구역 수 (`USE_FURNACE`일 때) | O | O |
| `CONTROL__SAME5` | 최근 5분 동안 내 구역 TC와 OP가 같은 방향으로 움직인 비율. 정상 제어면 반대 방향 (`USE_CONTROL`일 때) | O | O |

- 모든 계산은 같은 LOT 안에서, 현재보다 앞선 행만 사용함
- 공통 움직임의 \|Z\|도 7번 Z-score 기준과 같이 지난 값만으로 계산함
- 5분, 10분 전 값이 없는 LOT 앞부분은 0으로 둠 (컬럼을 켜고 꺼도 평가하는 행이 같도록)
- OP가 끝에 닿는 구역은 사실상 Z3뿐임 (OP-Z4 ~ Z6은 30 ~ 86% 안에서 움직임)

In [ ]:
features = []

for sensor in sensors:
    features.append(sensor)
    features.append(sensor + "__DIFF")

    if MINUTE_DATA:
        features = features + [sensor + "__MIN", sensor + "__MAX", sensor + "__STD"]
    else:
        features = features + [sensor + "__DEV60", sensor + "__STD60"]

lots = sorted(df["LOT_NO"].unique())

for lot in lots:
    rows = df["LOT_NO"] == lot

    for sensor in sensors:
        values = df.loc[rows, sensor]
        previous = values.shift(1)  # 바로 앞 행 값

        df.loc[rows, sensor + "__DIFF"] = values - previous

        if not MINUTE_DATA:
            df.loc[rows, sensor + "__DEV60"] = values - previous.rolling(60, min_periods=10).mean()
            df.loc[rows, sensor + "__STD60"] = previous.rolling(60, min_periods=10).std()

# 1분 안에 행이 하나뿐이면 표준편차가 비므로 0으로 채움
if MINUTE_DATA:
    for sensor in sensors:
        df[sensor + "__STD"] = df[sensor + "__STD"].fillna(0)

# 공통 움직임 컬럼: Z3 ~ Z6 온도가 최근 5분 사이에 같이 움직였는지
if USE_COMMON:
    WINDOW = 5 if MINUTE_DATA else 300  # 1분 데이터는 5행, 원기록은 300행 (둘 다 5분)

    for sensor in COMMON_SENSORS:
        features.append(sensor + "__CHG5")
    features.append("COMMON__OTHERS")

    df["COMMON__OTHERS"] = 0

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for sensor in COMMON_SENSORS:
            values = df.loc[rows, sensor]
            previous = values.shift(1)

            # 5분 전 대비 변화
            df.loc[rows, sensor + "__CHG5"] = (values - values.shift(WINDOW)).fillna(0)

            # 내 구역 온도는 세지 않음
            if sensor in sensors:
                continue

            # 지난 값만으로 |Z| > 3이었는지, 최근 5분 안에 한 번이라도
            past_mean = previous.expanding(min_periods=30).mean()
            past_std = previous.expanding(min_periods=30).std(ddof=0)
            past_std = past_std.replace(0, np.nan)

            over = ((values - past_mean).abs() / past_std > 3).astype(int)
            recent = over.rolling(WINDOW, min_periods=1).max()

            df.loc[rows, "COMMON__OTHERS"] = df.loc[rows, "COMMON__OTHERS"] + recent

# 1분 데이터는 10행, 원기록은 600행 (둘 다 10분)
WINDOW10 = 10 if MINUTE_DATA else 600

# OP 포화비율: 최근 10분 중 OP가 끝(99% 이상 또는 5% 이하)에 닿은 비율
if USE_SAT:
    op_sensors = []
    for sensor in sensors:
        if sensor.startswith("OP-"):
            op_sensors.append(sensor)
            features.append(sensor + "__SAT10")

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for sensor in op_sensors:
            if MINUTE_DATA:
                touched = (df.loc[rows, sensor + "__MAX"] >= 99) | (df.loc[rows, sensor + "__MIN"] <= 5)
            else:
                touched = (df.loc[rows, sensor] >= 99) | (df.loc[rows, sensor] <= 5)

            df.loc[rows, sensor + "__SAT10"] = touched.astype(int).rolling(WINDOW10, min_periods=1).mean()

# LOT 경과시간: 4번 셀에서 계산한 값을 입력에 넣음
if USE_ELAPSED:
    features.append("LOT__ELAPSED")

# 10분 이동통계: 직전 10분 평균과의 차이, 직전 10분 표준편차
if USE_ROLL10:
    for sensor in sensors:
        features = features + [sensor + "__DEV10", sensor + "__STD10"]

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for sensor in sensors:
            values = df.loc[rows, sensor]
            previous = values.shift(1)

            df.loc[rows, sensor + "__DEV10"] = (values - previous.rolling(WINDOW10, min_periods=2).mean()).fillna(0)
            df.loc[rows, sensor + "__STD10"] = previous.rolling(WINDOW10, min_periods=2).std().fillna(0)

# 변화율: 최근 5분 동안 1분에 평균 얼마나 변했는지
if USE_RATE:
    STEP = 5 if MINUTE_DATA else 300  # 5분 전 행

    for sensor in sensors:
        features.append(sensor + "__RATE5")

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for sensor in sensors:
            values = df.loc[rows, sensor]
            df.loc[rows, sensor + "__RATE5"] = ((values - values.shift(STEP)) / 5).fillna(0)

# 지난 값 |Z|: Z-score 기준선과 같은 정보를 RandomForest에도 줌
if USE_ZPAST:
    for sensor in sensors:
        features.append(sensor + "__ZPAST")
        if MINUTE_DATA:
            features.append(sensor + "__ZPAST_EDGE")

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for sensor in sensors:
            values = df.loc[rows, sensor]
            previous = values.shift(1)

            past_mean = previous.expanding(min_periods=30).mean()
            past_std = previous.expanding(min_periods=30).std(ddof=0)
            past_std = past_std.replace(0, np.nan)

            df.loc[rows, sensor + "__ZPAST"] = ((values - past_mean).abs() / past_std).fillna(0)

            # 1분 안의 최대, 최소 중 지난 평균에서 더 먼 쪽
            if MINUTE_DATA:
                high = (df.loc[rows, sensor + "__MAX"] - past_mean).abs()
                low = (df.loc[rows, sensor + "__MIN"] - past_mean).abs()
                edge = high.where(high > low, low)
                df.loc[rows, sensor + "__ZPAST_EDGE"] = (edge / past_std).fillna(0)

# OP 공통 움직임: Z3 ~ Z6 출력이 최근 5분 사이에 같이 움직였는지
if USE_OPCOMMON:
    WINDOW = 5 if MINUTE_DATA else 300

    for sensor in COMMON_OPS:
        features.append(sensor + "__CHG5")
    features.append("COMMON__OP_OTHERS")

    df["COMMON__OP_OTHERS"] = 0

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for sensor in COMMON_OPS:
            values = df.loc[rows, sensor]
            previous = values.shift(1)

            df.loc[rows, sensor + "__CHG5"] = (values - values.shift(WINDOW)).fillna(0)

            # 내 구역 출력은 세지 않음
            if sensor in sensors:
                continue

            past_mean = previous.expanding(min_periods=30).mean()
            past_std = previous.expanding(min_periods=30).std(ddof=0)
            past_std = past_std.replace(0, np.nan)

            over = ((values - past_mean).abs() / past_std > 3).astype(int)
            recent = over.rolling(WINDOW, min_periods=1).max()

            df.loc[rows, "COMMON__OP_OTHERS"] = df.loc[rows, "COMMON__OP_OTHERS"] + recent

# 로 전체 동시 이상 수: Z1 ~ Z8 중 내 구역 말고 최근 5분 안에 센서 하나라도 |Z| > 3이었던 구역 수
if USE_FURNACE:
    WINDOW = 5 if MINUTE_DATA else 300

    features.append("FURNACE__OTHERS")
    df["FURNACE__OTHERS"] = 0

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for zone in FURNACE_ZONES:
            if zone == ZONE:
                continue

            # 이 구역 센서 중 하나라도 넘었으면 1
            zone_over = pd.Series(0, index=df.index[rows])

            for sensor in GROUPS[zone]:
                values = df.loc[rows, sensor]
                previous = values.shift(1)

                past_mean = previous.expanding(min_periods=30).mean()
                past_std = previous.expanding(min_periods=30).std(ddof=0)
                past_std = past_std.replace(0, np.nan)

                over = ((values - past_mean).abs() / past_std > 3).astype(int)
                zone_over = (zone_over + over).clip(upper=1)

            recent = zone_over.rolling(WINDOW, min_periods=1).max()
            df.loc[rows, "FURNACE__OTHERS"] = df.loc[rows, "FURNACE__OTHERS"] + recent

# TC-OP 제어 어긋남: 최근 5분 동안 내 구역 TC와 OP가 같은 방향으로 움직인 비율
if USE_CONTROL:
    WINDOW = 5 if MINUTE_DATA else 300

    tc_sensor = None
    op_sensor = None
    for sensor in sensors:
        if sensor.startswith("TC-"):
            tc_sensor = sensor
        if sensor.startswith("OP-"):
            op_sensor = sensor

    # TC와 OP가 함께 있는 구역만 (CLN, Z9, COOLING은 OP가 없음)
    if tc_sensor is not None and op_sensor is not None:
        features.append("CONTROL__SAME5")

        for lot in lots:
            rows = df["LOT_NO"] == lot

            tc_change = df.loc[rows, tc_sensor].diff()
            op_change = df.loc[rows, op_sensor].diff()

            same = ((tc_change * op_change) > 0).astype(int)
            df.loc[rows, "CONTROL__SAME5"] = same.rolling(WINDOW, min_periods=1).mean()

# LOT 앞부분처럼 값이 빈 행은 뺌
before = len(df)
df = df.dropna(subset=features).reset_index(drop=True)

print("입력 변수:", features)
print("값이 비어 뺀 행:", before - len(df))

## 7. Z-score 기준 (지난 값만 사용)

- 같은 LOT에서 현재 행 이전까지 본 값의 평균과 표준편차로 \|Z\|를 계산함
- `expanding`은 LOT 처음부터 지금까지 (`rolling`은 최근 N개)
- 앞의 30행은 기준이 부족해서 0으로 둠
- 구역 센서 중 하나라도 \|Z\| > 3이면 1

In [ ]:
df["PRED_Z"] = 0

for lot in lots:
    rows = df["LOT_NO"] == lot

    for sensor in sensors:
        values = df.loc[rows, sensor]
        previous = values.shift(1)

        past_mean = previous.expanding(min_periods=30).mean()
        past_std = previous.expanding(min_periods=30).std(ddof=0)
        past_std = past_std.replace(0, np.nan)

        z = (values - past_mean).abs() / past_std

        over_index = z[z > 3].index
        df.loc[over_index, "PRED_Z"] = 1

print("Z-score가 1로 예측한 행:", int(df["PRED_Z"].sum()))

## 8. 시간 순서대로 학습하고 시험하기

| 시험 LOT | 학습 LOT |
| --- | --- |
| 5번째 LOT | 1 ~ 4번째 |
| 6번째 LOT | 1 ~ 5번째 |
| ... | ... |
| 14번째 LOT | 1 ~ 13번째 |

- IsolationForest: 학습 LOT의 LABEL 1 비율만큼을 이상으로 봄 (`contamination`)
- RandomForest: 1이 적어서 `class_weight="balanced"`로 1에 무게를 더 줌, 나무 깊이는 10까지
- 학습 LOT에 1이 하나도 없으면 두 모델은 0으로 둠

In [ ]:
FIRST_TEST = 4  # 앞의 4개 LOT은 학습에만 씀

df["TEST"] = False
df["PRED_DUMMY"] = 0
df["PRED_IF"] = 0
df["PRED_RF"] = 0

for k in range(FIRST_TEST, len(lots)):
    test_lot = lots[k]
    train_lots = lots[:k]

    train = df[df["LOT_NO"].isin(train_lots)]
    test_rows = df["LOT_NO"] == test_lot

    X_train = train[features]
    y_train = train["LABEL"]
    X_test = df.loc[test_rows, features]

    df.loc[test_rows, "TEST"] = True

    # Dummy: 전부 0
    dummy = DummyClassifier(strategy="constant", constant=0)
    dummy.fit(X_train, y_train)
    df.loc[test_rows, "PRED_DUMMY"] = dummy.predict(X_test)

    rate = y_train.mean()

    if rate > 0:
        # IsolationForest: -1(이상) -> 1, 1(정상) -> 0
        iso = IsolationForest(n_estimators=100, contamination=min(rate, 0.5), random_state=42, n_jobs=-1)
        iso.fit(X_train)
        df.loc[test_rows, "PRED_IF"] = (iso.predict(X_test) == -1).astype(int)

        # RandomForest
        rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)
        df.loc[test_rows, "PRED_RF"] = rf.predict(X_test)

    print("시험 LOT", test_lot, "/ 학습 LOT", len(train_lots), "개 / 학습 1 비율", round(rate * 100, 2), "%")

## 9. 전체 결과 (시험한 10개 LOT을 모아서)

| 칸 | 뜻 |
| --- | --- |
| TP | 1을 1로 맞힘 |
| FP | 0인데 1로 경보 (오탐) |
| FN | 1인데 놓침 (미탐) |
| TN | 0을 0으로 맞힘 |
| 정확도 | 참고용. 0이 대부분이라 Dummy도 높게 나옴 |

In [ ]:
MODELS = {
    "Dummy (전부 0)": "PRED_DUMMY",
    "Z-score (지난 값)": "PRED_Z",
    "IsolationForest": "PRED_IF",
    "RandomForest": "PRED_RF",
}

# 켠 컬럼 묶음을 결과 이름에 표시 (04에서 따로 보임)
SWITCHES = [
    [USE_COMMON, "공통 움직임", "_common"],
    [USE_SAT, "포화비율", "_sat"],
    [USE_ELAPSED, "경과시간", "_elapsed"],
    [USE_ROLL10, "10분 통계", "_roll10"],
    [USE_RATE, "변화율", "_rate"],
    [USE_ZPAST, "지난 값 Z", "_zpast"],
    [USE_OPCOMMON, "OP 공통 움직임", "_opcommon"],
    [USE_FURNACE, "로 전체 동시 이상", "_furnace"],
    [USE_CONTROL, "제어 어긋남", "_control"],
]

data_name = DATA_FILE
name_tail = ""
for on, label, tail in SWITCHES:
    if on:
        data_name = data_name + " + " + label
        name_tail = name_tail + tail

test = df[df["TEST"] == True]
y_true = test["LABEL"]

rows = []

for name in MODELS:
    y_pred = test[MODELS[name]]

    rows.append({
        "DATA_FILE": data_name,
        "ZONE": ZONE,
        "모델": name,
        "시험 행": len(test),
        "LABEL 1": int(y_true.sum()),
        "TP": int(((y_true == 1) & (y_pred == 1)).sum()),
        "FP": int(((y_true == 0) & (y_pred == 1)).sum()),
        "FN": int(((y_true == 1) & (y_pred == 0)).sum()),
        "TN": int(((y_true == 0) & (y_pred == 0)).sum()),
        "정확도": round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_true, y_pred, zero_division=0), 4),
    })

summary = pd.DataFrame(rows)
summary

## 10. 시험 LOT별 F1 (LABEL 1이 없는 LOT은 빈칸)

In [ ]:
rows = []

for lot in lots[FIRST_TEST:]:
    part = test[test["LOT_NO"] == lot]
    row = {"LOT_NO": lot, "시험 행": len(part), "LABEL 1": int(part["LABEL"].sum())}

    for name in MODELS:
        if part["LABEL"].sum() > 0:
            row[name] = round(f1_score(part["LABEL"], part[MODELS[name]], zero_division=0), 4)
        else:
            row[name] = np.nan

    rows.append(row)

by_lot = pd.DataFrame(rows)
by_lot

## 11. 저장

| 파일 | 내용 |
| --- | --- |
| `파일명_구역_summary.csv` | 9번 전체 결과 |
| `파일명_구역_by_lot.csv` | 10번 LOT별 F1 |
| `파일명_구역_predictions.csv` | 시험 행의 정답과 네 모델 예측 (오탐, 미탐 찾을 때 사용) |

- 켠 컬럼 묶음에 따라 이름 끝에 `_common`, `_sat`, `_elapsed`, `_roll10`, `_rate`, `_zpast`, `_opcommon`, `_furnace`, `_control`이 붙음

In [ ]:
name = DATA_FILE.replace(".csv", "") + "_" + ZONE + name_tail

summary.to_csv(os.path.join(RESULT_DIR, name + "_summary.csv"), index=False, encoding=ENC)
by_lot.to_csv(os.path.join(RESULT_DIR, name + "_by_lot.csv"), index=False, encoding=ENC)

prediction_columns = ["LOT_NO", "MEAS_DT", "LABEL", "PRED_DUMMY", "PRED_Z", "PRED_IF", "PRED_RF"]
test[prediction_columns].to_csv(os.path.join(RESULT_DIR, name + "_predictions.csv"), index=False, encoding=ENC)

print("저장:", name)